<a href="https://colab.research.google.com/github/amakalarry/Synthetic-Crime-Scene-Research/blob/main/vertex_to_yolov11_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 Vertex JSONL ➜ YOLOv11 Full Pipeline

End‑to‑end notebook that:
1) Parses **Google Vertex** `jsonl` annotations (normalized x/y mins & maxes)
2) Converts to **YOLO** label files per split (train/val/test)
3) Builds a correct `data.yaml`
4) Trains **YOLOv11** (falls back to YOLOv8 if needed)
5) Evaluates on val & test
6) Runs inference on test images and saves predictions, labels, and a ZIP


In [1]:
!pip -q install ultralytics PyYAML tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 13.8 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ✍️ Paths — update if needed
VERTEX_JSONL = "/content/drive/MyDrive/yolo_dataset_v2/vertex_annotations.jsonl"
DATASET_ROOT = "/content/drive/MyDrive/yolo_dataset_v2"  # contains train/ val/ test/ each with images/

# splits
SPLITS = ["train", "val", "test"]
IMG_SUBDIR = "images"  # expected under each split
LBL_SUBDIR = "labels"  # will be created under each split


In [5]:
import os, json
from tqdm import tqdm

def ensure_dirs():
    for split in SPLITS:
        img_dir = os.path.join(DATASET_ROOT, split, IMG_SUBDIR)
        lbl_dir = os.path.join(DATASET_ROOT, split, LBL_SUBDIR)
        if not os.path.exists(img_dir):
            raise FileNotFoundError(f"Missing images dir: {img_dir}")
        os.makedirs(lbl_dir, exist_ok=True)

def parse_vertex_jsonl(jsonl_path):
    mapping = {}
    classes = []  # preserve first-seen order
    with open(jsonl_path, 'r') as f:
        for line in f:
            line=line.strip()
            if not line:
                continue
            obj = json.loads(line)
            uri = obj.get('imageGcsUri') or obj.get('imageUri') or obj.get('imagePath')
            if not uri:
                continue
            filename = os.path.basename(uri)
            anns = obj.get('boundingBoxAnnotations', [])
            for ann in anns:
                name = ann.get('displayName') or ann.get('label') or ann.get('class')
                xMin = float(ann['xMin']); xMax = float(ann['xMax']); yMin = float(ann['yMin']); yMax = float(ann['yMax'])
                if name is None:
                    continue
                if name not in classes:
                    classes.append(name)
                mapping.setdefault(filename, []).append((name, xMin, xMax, yMin, yMax))
    return mapping, classes

def write_yolo_labels(vertex_map, classes):
    name_to_id = {n:i for i,n in enumerate(classes)}
    written = 0
    for split in SPLITS:
        img_dir = os.path.join(DATASET_ROOT, split, IMG_SUBDIR)
        lbl_dir = os.path.join(DATASET_ROOT, split, LBL_SUBDIR)
        os.makedirs(lbl_dir, exist_ok=True)
        for fname, anns in vertex_map.items():
            img_path = os.path.join(img_dir, fname)
            if not os.path.exists(img_path):
                continue
            lines = []
            for (name, xMin, xMax, yMin, yMax) in anns:
                cx = (xMin + xMax) / 2.0
                cy = (yMin + yMax) / 2.0
                w  = (xMax - xMin)
                h  = (yMax - yMin)
                cls_id = name_to_id[name]
                cx = max(0.0, min(1.0, cx)); cy = max(0.0, min(1.0, cy))
                w = max(0.0, min(1.0, w)); h = max(0.0, min(1.0, h))
                lines.append(f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
            label_path = os.path.join(lbl_dir, os.path.splitext(fname)[0] + ".txt")
            with open(label_path, 'w') as lf:
                lf.write("\n".join(lines))
            written += 1
    return written, {n:i for i,n in enumerate(classes)}

def create_data_yaml(classes):
    import yaml
    names_dict = {i:n for i,n in enumerate(classes)}
    data = {
        'path': DATASET_ROOT,
        'train': os.path.join(DATASET_ROOT, 'train', IMG_SUBDIR),
        'val': os.path.join(DATASET_ROOT, 'val', IMG_SUBDIR),
        'test': os.path.join(DATASET_ROOT, 'test', IMG_SUBDIR),
        'names': names_dict,
        'nc': len(classes)
    }
    yaml_path = os.path.join(DATASET_ROOT, 'data.yaml')
    with open(yaml_path, 'w') as f:
        yaml.dump(data, f, sort_keys=False)
    return yaml_path

ensure_dirs()
vertex_map, classes = parse_vertex_jsonl(VERTEX_JSONL)
print(f"Found {len(classes)} classes: {classes}")
written, name_to_id = write_yolo_labels(vertex_map, classes)
print(f"Wrote labels for {written} images across splits.")
yaml_path = create_data_yaml(classes)
print(f"data.yaml created at: {yaml_path}")

Found 13 classes: ['Body', 'Laptop', 'Mobilephone', 'GlassCup', 'Tablet', 'Smartwatch', 'Keyboard', 'flashdrive', 'Smartspeaker', 'Router', 'TV', 'Knife', 'Camera']
Wrote labels for 372 images across splits.
data.yaml created at: /content/drive/MyDrive/yolo_dataset_v2/data.yaml


In [6]:
from ultralytics import YOLO
import torch, os

weights_tried = []
model = None
for w in ["yolo11n.pt", "yolov11n.pt", "yolov8n.pt"]:
    try:
        model = YOLO(w)
        print(f"Loaded base weights: {w}")
        break
    except Exception as e:
        weights_tried.append((w, str(e)))
        continue
if model is None:
    raise RuntimeError(f"Could not load YOLO base weights. Tried: {weights_tried}")

device = '0' if torch.cuda.is_available() else 'cpu'
print("Training on device:", device)

results = model.train(
    data=os.path.join(DATASET_ROOT, 'data.yaml'),
    epochs=50,
    imgsz=640,
    batch=16 if device!='cpu' else 8,
    patience=20,
    project=os.path.join(DATASET_ROOT, 'runs', 'train'),
    name='vertex_yolo11',
    device=device,
    verbose=True
)
print("Training done.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loaded base weights: yolo11n.pt
Training on device: cpu
Ultralytics 8.3.213 🚀 Python-3.12.12 torch-2.8.0+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/yolo_dataset_v2/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=Fals

In [7]:
best_weights = os.path.join(DATASET_ROOT, 'runs', 'train', 'vertex_yolo11', 'weights', 'best.pt')
print("Best weights:", best_weights)
eval_model = YOLO(best_weights)
val_metrics = eval_model.val(data=os.path.join(DATASET_ROOT, 'data.yaml'), split='val', imgsz=640, batch=16, plots=True)
test_metrics = eval_model.val(data=os.path.join(DATASET_ROOT, 'data.yaml'), split='test', imgsz=640, batch=16, plots=True)
print('Val mAP@0.5:', getattr(val_metrics.box, 'map50', None))
print('Test mAP@0.5:', getattr(test_metrics.box, 'map50', None))

Best weights: /content/drive/MyDrive/yolo_dataset_v2/runs/train/vertex_yolo11/weights/best.pt
Ultralytics 8.3.213 🚀 Python-3.12.12 torch-2.8.0+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11n summary (fused): 100 layers, 2,584,687 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.7±0.3 ms, read: 38.7±14.6 MB/s, size: 162.4 KB)
val: Scanning /content/drive/MyDrive/yolo_dataset_v2/val/labels.cache... 80 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 80/80 105.1Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 0.3it/s 14.9s
                   all         80        322      0.759      0.798      0.762      0.487
                  Body         14         14      0.778      0.857       0.75      0.459
                Laptop         48         50      0.824      0.939      0.943      0.661
           Mobilephone         48         58      0.756      0.897       0.82       0.53
              GlassCup 

In [9]:
# ------------------------------------------------------------
# 📊 Evaluate best model on val and test sets
# ------------------------------------------------------------
best_weights = os.path.join(DATASET_ROOT, 'runs', 'train', 'vertex_yolo11', 'weights', 'best.pt')
print("Best weights:", best_weights)

eval_model = YOLO(best_weights)

# Evaluation on validation and test sets
val_metrics = eval_model.val(
    data=os.path.join(DATASET_ROOT, 'data.yaml'),
    split='val',
    imgsz=640,
    batch=16,
    plots=True,
    project=os.path.join(DATASET_ROOT, 'runs', 'val_results'),
    name='val_eval'
)

test_metrics = eval_model.val(
    data=os.path.join(DATASET_ROOT, 'data.yaml'),
    split='test',
    imgsz=640,
    batch=16,
    plots=True,
    project=os.path.join(DATASET_ROOT, 'runs', 'test_results'),
    name='test_eval'
)

print('Val mAP@0.5:', getattr(val_metrics.box, 'map50', None))
print('Test mAP@0.5:', getattr(test_metrics.box, 'map50', None))


# ------------------------------------------------------------
# 🔎 Run inference and save annotated results inside yolo_dataset_v2
# ------------------------------------------------------------
out_project = os.path.join(DATASET_ROOT, 'runs', 'detect')
run_name = 'vertex_yolo11_test_infer'

pred = eval_model.predict(
    source=os.path.join(DATASET_ROOT, 'test', 'images'),
    conf=0.25,
    save=True,
    save_txt=True,
    save_conf=True,
    imgsz=640,
    project=out_project,
    name=run_name,
    exist_ok=True
)

print('✅ Inference completed. Results saved under:', os.path.join(out_project, run_name))


# ------------------------------------------------------------
# 📦 ZIP the inference folder and save it to your Drive
# ------------------------------------------------------------
import subprocess, shlex

pred_dir = os.path.join(out_project, run_name)
zip_path = os.path.join(DATASET_ROOT, 'vertex_yolo11_predictions.zip')

if os.path.exists(pred_dir):
    cmd = f"zip -r {zip_path} {shlex.quote(pred_dir)}"
    _ = subprocess.getoutput(cmd)
    print('📦 ZIP archive created at:', zip_path)
else:
    print('⚠️ Prediction folder not found:', pred_dir)


Best weights: /content/drive/MyDrive/yolo_dataset_v2/runs/train/vertex_yolo11/weights/best.pt
Ultralytics 8.3.213 🚀 Python-3.12.12 torch-2.8.0+cu126 CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11n summary (fused): 100 layers, 2,584,687 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.6±0.2 ms, read: 24.7±12.8 MB/s, size: 157.3 KB)
val: Scanning /content/drive/MyDrive/yolo_dataset_v2/val/labels.cache... 80 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 80/80 69.6Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 0.3it/s 16.5s
                   all         80        322      0.759      0.798      0.762      0.487
                  Body         14         14      0.778      0.857       0.75      0.459
                Laptop         48         50      0.824      0.939      0.943      0.661
           Mobilephone         48         58      0.756      0.897       0.82       0.53
              GlassCup  

In [8]:
out_project = os.path.join(DATASET_ROOT, 'runs', 'detect')
run_name = 'vertex_yolo11_test_infer'
pred = eval_model.predict(
    source=os.path.join(DATASET_ROOT, 'test', 'images'),
    conf=0.25,
    save=True,
    save_txt=True,
    save_conf=True,
    imgsz=640,
    project=out_project,
    name=run_name,
    exist_ok=True
)
print('Inference saved under:', os.path.join(out_project, run_name))


image 1/40 /content/drive/MyDrive/yolo_dataset_v2/test/images/ai_video01_frame_0.jpg: 384x640 1 Laptop, 1 Mobilephone, 1 Keyboard, 1 flashdrive, 175.8ms
image 2/40 /content/drive/MyDrive/yolo_dataset_v2/test/images/ai_video02_frame_120.jpg: 384x640 1 Laptop, 1 Mobilephone, 1 Tablet, 1 flashdrive, 134.9ms
image 3/40 /content/drive/MyDrive/yolo_dataset_v2/test/images/ai_video04_frame_96.jpg: 384x640 1 Laptop, 1 Mobilephone, 1 Tablet, 1 Smartwatch, 1 flashdrive, 1 Smartspeaker, 138.3ms
image 4/40 /content/drive/MyDrive/yolo_dataset_v2/test/images/ai_video05_frame_96.jpg: 384x640 1 Laptop, 1 Tablet, 1 Smartwatch, 137.3ms
image 5/40 /content/drive/MyDrive/yolo_dataset_v2/test/images/ai_video06_frame_144.jpg: 384x640 1 Mobilephone, 1 Tablet, 2 Smartwatchs, 3 flashdrives, 1 Smartspeaker, 1 Router, 1 TV, 145.8ms
image 6/40 /content/drive/MyDrive/yolo_dataset_v2/test/images/ai_video06_frame_48.jpg: 384x640 1 Mobilephone, 1 Tablet, 2 Smartwatchs, 4 flashdrives, 1 Smartspeaker, 1 Router, 1 TV, 1

In [ ]:
import subprocess, shlex, os
pred_dir = os.path.join(DATASET_ROOT, 'runs', 'detect', 'vertex_yolo11_test_infer')
zip_path = '/content/vertex_yolo11_predictions.zip'
if os.path.exists(pred_dir):
    cmd = f"zip -r {zip_path} {shlex.quote(pred_dir)}"
    _ = subprocess.getoutput(cmd)
    print('ZIP ready at:', zip_path)
else:
    print('Prediction folder not found:', pred_dir)